# Train a segmentation model with DINOv3
---
In this notebook, we will create a foreground segmentor to segment tactile pavings by training a custom regression classifier on our custom dataset using pre-trained DINOv3 weights.

Using the trained model, we can then run inference on our test dataset of real tactile paving data to get a picture of how the model's segmentations would work in real life scenarios.

This notebook is based off the forground segmentation notebook provided by Meta here: [link](https://github.com/facebookresearch/dinov3/blob/main/notebooks/foreground_segmentation.ipynb)

## Data

Please request permission access from the forms provided to download both our custom synthetically augmented dataset as well as our collected real-world testset that will be used for training and evalutation respectively.

In order to train the model, the datasets must be arranged in a specific format. For DINOv3 to extract the features this is the recommended structure that will be used:
```
my_data_dir
├── images
│   ├── image0.jpg
│   └── image1.jpg
└── masks
    ├── image0.png
    └── image1.png
```   

where each jpg image has a corresponding png mask with the same filename. 

We can load the data in memory so that it can be used by the vision transformer to extract the flattened featuers.

In [ ]:
import io
import os
import pickle
import tarfile
import urllib

from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score
from sklearn.linear_model import LogisticRegression
import torch
import torchvision.transforms.functional as TF
from tqdm import tqdm

IMAGES_URI = "/..." # Path to images
LABELS_URI = "/..." # path to masks

def load_images(uri: str) -> list[Image.Image]:
    images = []
    for filename in os.listdir(uri):
        if filename.endswith('.jpg') or filename.endswith('.png'):
            image = Image.open(os.path.join(uri, filename))
            images.append(image)
    return images
    
images = load_images(IMAGES_URI)
labels = load_images(LABELS_URI)
n_images = len(images)
assert n_images == len(labels), f"{len(images)=}, {len(labels)=}"

print(f"Loaded {n_images} images and labels")

## Setup

It is highly recommended to install Pytorch with CUDA support as they are the required dependencies for loading the model. All extensive detail can be accesses at the official repo: [Meta - DINOv3](https://github.com/facebookresearch/dinov3)

Clone the repository:

In [ ]:
!git clone https://github.com/facebookresearch/dinov3.git

To use DINOV3, please sign up and accept the terms of use from Meta to get access to the DINOv3 checkpoints and their download links. 

We can set our DINOV3 model to use here. For the purpose of this, we are goning to use the 'DINOV3-S' i.e. small model.


In [ ]:
REPO_DIR = "..." # Path where repo was cloned
DINOv3_model_url = "..." # Add the download link here

In [ ]:
# examples of available DINOv3 models:
MODEL_DINOV3_VITS = "dinov3_vits16"
MODEL_DINOV3_VITSP = "dinov3_vits16plus"
MODEL_DINOV3_VITB = "dinov3_vitb16"
MODEL_DINOV3_VITL = "dinov3_vitl16"
MODEL_DINOV3_VITHP = "dinov3_vith16plus"
MODEL_DINOV3_VIT7B = "dinov3_vit7b16"

MODEL_NAME = MODEL_DINOV3_VITS


# model = torch.hub.load(
#     repo_or_dir=DINOV3_LOCATION,
#     model=MODEL_NAME,
#     source="local" if DINOV3_LOCATION != DINOV3_GITHUB_LOCATION else "github",
# )

model = torch.hub.load(REPO_DIR, MODEL_NAME, source='local', weights=DINOv3_model_url)
model.cuda()

### Building Per-Patch Label Map

Since our models run with a patch size of 16, we have to quantize the ground truth to a 16x16 pixels grid. To achieve this, we define:
- the resize transform to resize an image such that it aligns well with the 16x16 grid;
- a uniform 16x16 conv layer as a [box blur filter](https://en.wikipedia.org/wiki/Box_blur) with stride equal to the patch size.

In [ ]:
PATCH_SIZE = 16
IMAGE_SIZE = 768

# quantization filter for the given patch size
patch_quant_filter = torch.nn.Conv2d(1, 1, PATCH_SIZE, stride=PATCH_SIZE, bias=False)
patch_quant_filter.weight.data.fill_(1.0 / (PATCH_SIZE * PATCH_SIZE))

# image resize transform to dimensions divisible by patch size
def resize_transform(
    mask_image: Image,
    image_size: int = IMAGE_SIZE,
    patch_size: int = PATCH_SIZE,
) -> torch.Tensor:
    w, h = mask_image.size
    h_patches = int(image_size / patch_size)
    w_patches = int((w * image_size) / (h * patch_size))
    return TF.to_tensor(TF.resize(mask_image, (h_patches * patch_size, w_patches * patch_size)))

### Extracting Features and Labels for All the Images
Now we will run a dense feature extraction with our model and our data by iterating through all the training images, and extract for each image the patch labels, as well as the patch features.

In [ ]:
xs = []
ys = []
image_index = []

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

MODEL_TO_NUM_LAYERS = {
    MODEL_DINOV3_VITS: 12,
    MODEL_DINOV3_VITSP: 12,
    MODEL_DINOV3_VITB: 12,
    MODEL_DINOV3_VITL: 24,
    MODEL_DINOV3_VITHP: 32,
    MODEL_DINOV3_VIT7B: 40,
}

n_layers = MODEL_TO_NUM_LAYERS[MODEL_NAME]

with torch.inference_mode():
    with torch.autocast(device_type='cuda', dtype=torch.float32):
        for i in tqdm(range(n_images), desc="Processing images"):
            # Loading the ground truth
            mask_i = labels[i].split()[-1]
            mask_i_resized = resize_transform(mask_i)
            mask_i_quantized = patch_quant_filter(mask_i_resized).squeeze().view(-1).detach().cpu()
            ys.append(mask_i_quantized)
            # Loading the image data 
            image_i = images[i].convert('RGB')
            image_i_resized = resize_transform(image_i)
            image_i_resized = TF.normalize(image_i_resized, mean=IMAGENET_MEAN, std=IMAGENET_STD)
            image_i_resized = image_i_resized.unsqueeze(0).cuda()

            feats = model.get_intermediate_layers(image_i_resized, n=range(n_layers), reshape=True, norm=True)
            dim = feats[-1].shape[1]
            xs.append(feats[-1].squeeze().view(dim, -1).permute(1,0).detach().cpu())

            image_index.append(i * torch.ones(ys[-1].shape))


# Concatenate all lists into torch tensors 
xs = torch.cat(xs)
ys = torch.cat(ys)
image_index = torch.cat(image_index)

# keeping only the patches that have clear positive or negative label
idx = (ys < 0.01) | (ys > 0.99)
xs = xs[idx]
ys = ys[idx]
image_index = image_index[idx]

print("Design matrix of size : ", xs.shape)
print("Label matrix of size : ", ys.shape)

**Note:**
- This is quite a memory intensive process (even for the 'small' model we are using). It is recommended to use a cluster if possible to use multiple resources and workers together to avoid crashing.

## Training

Now that we've computed all the features, we can use these and train a regression classifier to segment the tactile pavings in an image.

This works as a classic foreground-background solver due to the nature of our data and task. We will simply use the Logistic Regression class from the scikit-learn library to accomplish this:

**Note:**
- There is an optimal value of 'C' that can be set by experimenting across values from 0.1 to 0.9. Feel free to do so however for our case, we deemed 0.1 to be a satisfactory value.

In [ ]:
clf = LogisticRegression(random_state=0, C=0.1, max_iter=100000, verbose=2).fit(xs.numpy(), (ys > 0).long().numpy())

In [ ]:
save_root = "..."   # save path 
model_path = os.path.join(save_root, "...") # name of classifer 
with open(model_path, "wb") as f:
  pickle.dump(clf, f)

## Inference

Now that we have a trained classifier, we can use it to predict the probability of a patch being a tactile paving by segmenting it as the 'foreground' vs the 'background' of the image.


First, load the model

In [ ]:
file_path = "" # Replace with your checkpoint file path

with open(file_path, 'rb') as file:
    clf = pickle.load(file)

print('Model loaded successfully.')

We can now use a sample image from our test set and produce a figure score i.e. predicted mask for the detected tactile paving. 

We can evaluate the accuracy of it by comparing it with the ground truth mask of the test image. 

In [ ]:

test_image_fpath = "..." # test image path
test_mask_fpath = "..." # test image mask path

test_image= Image.open(test_image_fpath)
test_image_resized = resize_transform(test_image)
test_image_normalized = TF.normalize(test_image_resized, mean=IMAGENET_MEAN, std=IMAGENET_STD) # Quantize the image

test_mask= Image.open(test_mask_fpath)

# Extract features
with torch.inference_mode():
    with torch.autocast(device_type='cuda', dtype=torch.float32):
        feats = model.get_intermediate_layers(test_image_normalized.unsqueeze(0).cuda(), n=range(n_layers), reshape=True, norm=True)
        x = feats[-1].squeeze().detach().cpu()
        dim = x.shape[0]
        x = x.view(dim, -1).permute(1, 0)

h_patches, w_patches = [int(d / PATCH_SIZE) for d in test_image_resized.shape[1:]]

# Pass feauteres into classifier 
fg_score = clf.predict_proba(x)[:, 1].reshape(h_patches, w_patches)
fg_score_mf = torch.from_numpy(signal.medfilt2d(fg_score, kernel_size=3))

threshold_value = 0.5  # Confidence value

boolean_mask = np.asarray(fg_score) > threshold_value  # Pixels greater than threshold_value become True
binary_mask_0_255 = boolean_mask.astype(np.uint8) * 255

pred_binary_image = Image.fromarray(binary_mask_0_255)

plt.figure(figsize=(9, 3), dpi=300)
plt.subplot(1, 3, 1)
plt.axis('off')
plt.imshow(test_image_resized.permute(1, 2, 0))
plt.title('input image')
plt.subplot(1, 3, 2)
plt.axis('off')
plt.imshow(binary_mask_0_255)
plt.title('prediction')
plt.subplot(1, 3, 3)
plt.axis('off')
# plt.imshow(fg_score_mf)
# plt.title('+ median filter')
plt.imshow(test_mask)
plt.title("ground truth")

plt.show()
